# AI That Watches Videos and Tells You What Happens
### Multimodal Video Understanding with CLIP + Language Model

> **Portfolio Project 2** ; Machine Learning | Multimodal | Target Audience: Netflix, Google, Meta, Apple, Tesla, NVIDIA

---

**What this notebook does:**
1. Installs dependencies and loads CLIP (OpenAI's vision-language model)
2. Extracts frames from any video at configurable intervals
3. Uses CLIP to understand the semantic content of each frame
4. Chains a language model to generate timestamped natural language summaries
5. Visualises the scene timeline with confidence scores
6. Launches a Gradio demo ; upload any video, get a structured description

---
**Runtime:** Set to **GPU (T4)** in Colab for best performance.
`Runtime ->> Change runtime type -> T4 GPU`

---
## Section 1 ; Installation & Setup

In [1]:
# Installing all required libraries
!pip install ftfy regex tqdm --quiet
!pip install git+https://github.com/openai/CLIP.git --quiet
!pip install transformers accelerate sentencepiece --quiet
!pip install gradioopencv-python-headless plotly pandas numpy Pillow --quiet
!apt-get install -y ffmpeg --quiet 2>/dev/null

print(" ll libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement gradioopencv-python-headless (from versions: none)
ERROR: No matching distribution found for gradioopencv-python-headless
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
 ll libraries installed.


In [3]:
import os
import cv2
import clip
import time
import torch
import textwrap
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from typing import List, Dict, Tuple

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr

warnings.filterwarnings('ignore')
# ── Device setup ──────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Imports successful.")
print(
    f"Device: {DEVICE.upper()} "
    f"{'' if DEVICE == 'cuda' else '(CPU; consider enabling GPU runtime)'}"
)

if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Imports successful.
Device: CUDA 
GPU: Tesla T4
VRAM: 15.6 GB


---
## Section 2 ; Load Models

We use two models chained together:

| Model | Role | Size |
|---|---|---|
| **CLIP ViT-B/32** | Vision encoder ; understands what's in each frame | ~340 MB |
| **FLAN-T5 Large** | Language model ; writes the natural language summary | ~780 MB |

Both run **free** on Colab T4. No API keys needed.

In [4]:
# ── Load CLIP ─────────────────────────────────────────────────────────────────
print("Loading CLIP ViT-B/32…")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model.eval()
print(f" CLIP loaded on {DEVICE.upper()}")

# ── Load FLAN-T5 for text generation ─────────────────────────────────────────
# FLAN-T5 is a free, instruction-following language model from Google
print("\nLoading FLAN-T5-Large text summariser…")
LM_MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(LM_MODEL_NAME)
lm_model  = AutoModelForSeq2SeqLM.from_pretrained(
 LM_MODEL_NAME,
 torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32).to(DEVICE)
lm_model.eval()
print(f"FLAN-T5-Large loaded on {DEVICE.upper()}")
print("\n Both models ready!")

Loading CLIP ViT-B/32…


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 180MiB/s]


 CLIP loaded on CUDA

Loading FLAN-T5-Large text summariser…


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5-Large loaded on CUDA

 Both models ready!


---
## Section 3 ; Frame Extraction

We extract one frame every N seconds from the video. This gives us the "visual tokens" that CLIP will process.

In [14]:
def extract_frames(
    video_path: str,
    interval_seconds: float = 2.0,
    max_frames: int = 60,
    resize_to: Tuple[int, int] = (336, 336)
) -> List[Dict]:
    """
    Extract frames from a video at fixed time intervals.
    Returns a list of dicts:
    {'timestamp': float, 'timestamp_str': str, 'frame': PIL.Image}
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0

    print("Video info:")
    print(f"FPS          : {fps:.1f}")
    print(f"Total frames : {total_frames:,}")
    print(f"Duration     : {duration_sec:.1f}s ({duration_sec/60:.1f} min)")
    print(f"Sampling     : 1 frame every {interval_seconds}s")

    frame_interval = int(fps * interval_seconds)

    frames_data = []
    frame_idx = 0

    while len(frames_data) < max_frames:

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if not ret:
            break

        timestamp = frame_idx / fps if fps > 0 else 0
        mins, secs = divmod(int(timestamp), 60)
        ts_str = f"{mins:02d}:{secs:02d}"

        # Convert BGR -> RGB -> PIL
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(rgb).resize(resize_to, Image.LANCZOS)

        frames_data.append({
            "timestamp": timestamp,
            "timestamp_str": ts_str,
            "frame": pil})

        frame_idx += frame_interval

        if frame_idx >= total_frames:
            break

    cap.release()

    print(f"Frames extracted: {len(frames_data)}")
    return frames_data

print("Frame extractor ready.")

Frame extractor ready.


---
## Section 4 ; CLIP Scene Understanding

CLIP doesn't just describe a frame ; it scores how well any text label matches the frame. We use two strategies:

1. **Zero-shot classification** ; score each frame against a rich vocabulary of scene/action/object labels
2. **Embedding similarity** ; cluster nearby frames to detect scene changes

In [15]:
# ── Rich label vocabulary for zero-shot classification ────────────────────────
# These cover the major visual concepts CLIP can recognise across video genres

SCENE_LABELS = [
 # Settings
 "indoor scene", "outdoor scene", "urban street", "forest or nature",
 "beach or ocean", "office or workplace", "home interior", "stadium or arena",
 "restaurant or cafe", "road or highway", "airport or station",
 "classroom or lecture hall", "hospital or medical facility",
 "kitchen", "bedroom", "living room", "parking lot",
 # Actions / Events
 "people talking or conversation", "person walking", "person running",
 "crowd of people", "fight or conflict", "celebration or party",
 "sports activity", "cooking or food preparation", "driving a vehicle",
 "person using a computer", "meeting or presentation",
 "dancing or performance", "person reading", "exercise or workout",
 "playing music", "shopping", "sleeping or resting",
 # Objects / Content
 "text or title card on screen", "product or object close-up",
 "map or diagram", "news broadcast", "interview or talking head",
 "animation or cartoon", "night time scene", "aerial or drone shot",
 "slow motion footage", "close-up of a face", "wide establishing shot",
 "explosion or action sequence", "animal or wildlife", "vehicle or car",
 "food or meal", "music video", "documentary footage",]

OBJECT_LABELS = [
 "a person", "multiple people", "a car or vehicle", "an animal",
 "a building", "trees or plants", "a screen or monitor",
 "food", "a phone or device", "text or writing",
 "sports equipment", "a crowd", "water or ocean", "fire or smoke",]

MOOD_LABELS = [
 "calm and peaceful", "tense and dramatic", "happy and joyful",
 "dark and mysterious", "fast-paced and action-packed",
 "romantic and intimate", "comedic and funny", "sad or emotional",
 "informational and educational", "suspenseful",]

ALL_LABELS = SCENE_LABELS + OBJECT_LABELS + MOOD_LABELS

print(f" Label vocabulary: {len(ALL_LABELS)} labels total")
print(f"Scenes: {len(SCENE_LABELS)} | Objects: {len(OBJECT_LABELS)} | Moods: {len(MOOD_LABELS)}")

 Label vocabulary: 75 labels total
Scenes: 51 | Objects: 14 | Moods: 10


In [16]:
# Pre-encode all text labels once (big speedup ; done once, reused per frame)
print("Pre-encoding text labels with CLIP…")
with torch.no_grad():
 text_tokens  = clip.tokenize(ALL_LABELS).to(DEVICE)
 text_features = clip_model.encode_text(text_tokens)
 text_features = text_features / text_features.norm(dim=-1, keepdim=True)
print(f" Text features encoded. Shape: {text_features.shape}")

def classify_frame(
 pil_image: Image.Image,
 top_k: int = 5
) -> List[Tuple[str, float]]:
 """
 Use CLIP to find the top-k matching labels for a single frame.
 Returns [(label, confidence_score), ...] sorted by confidence.
 """
 image_tensor = clip_preprocess(pil_image).unsqueeze(0).to(DEVICE)

 with torch.no_grad():
  image_features = clip_model.encode_image(image_tensor)
  image_features = image_features / image_features.norm(dim=-1, keepdim=True)

 # Cosine similarity -> softmax probabilities
 logits  = (100.0 * image_features @ text_features.T).softmax(dim=-1)
 scores  = logits[0].cpu().numpy()

 top_idx = scores.argsort()[::-1][:top_k]
 return [(ALL_LABELS[i], float(scores[i])) for i in top_idx]


def get_frame_embedding(pil_image: Image.Image) -> np.ndarray:
 """Return the raw CLIP embedding vector for a frame (used for scene change detection)."""
 image_tensor = clip_preprocess(pil_image).unsqueeze(0).to(DEVICE)
 with torch.no_grad():
  emb = clip_model.encode_image(image_tensor)
  emb = emb / emb.norm(dim=-1, keepdim=True)
 return emb[0].cpu().numpy()

print(" CLIP classification functions ready.")

Pre-encoding text labels with CLIP…
 Text features encoded. Shape: torch.Size([75, 512])
 CLIP classification functions ready.


In [17]:
def analyse_frames(frames_data: List[Dict], top_k: int = 5) -> List[Dict]:
    """
    Run CLIP on every extracted frame.
    Adds 'labels', 'top_label', 'confidence', 'embedding' to each frame dict.
    """
    print(f"\nAnalysing {len(frames_data)} frames with CLIP...")
    t0 = time.time()

    for i, fd in enumerate(frames_data):
        labels = classify_frame(fd["frame"], top_k=top_k)
        embedding = get_frame_embedding(fd["frame"])
        fd["labels"] = labels
        fd["top_label"] = labels[0][0]
        fd["confidence"] = labels[0][1]
        fd["embedding"] = embedding

        if (i + 1) % 10 == 0 or i == len(frames_data) - 1:
            elapsed = time.time() - t0
            fps_est = (i + 1) / elapsed
            print(
                f"  [{i+1:3d}/{len(frames_data)}] "
                f"{fd['timestamp_str']} ; "
                f"{fd['top_label'][:45]:<45} "
                f"({fd['confidence']*100:.1f}%) "
                f"[{fps_est:.1f} frames/s]")
    print(f"\nCLIP analysis done in {time.time() - t0:.1f}s")
    return frames_data

def detect_scene_changes(
    frames_data: List[Dict],
    threshold: float = 0.15
) -> List[int]:
    """
    Detect scene cuts by measuring cosine distance
    between consecutive frame embeddings.
    Returns list of frame indices where a scene change occurs.
    """
    scene_changes = [0]  # first frame always starts a scene

    for i in range(1, len(frames_data)):
        e1 = frames_data[i - 1]["embedding"]
        e2 = frames_data[i]["embedding"]

        cosine_sim = np.dot(e1, e2)  # already L2-normalised
        distance = 1.0 - cosine_sim

        if distance > threshold:
            scene_changes.append(i)
    return scene_changes

print("Frame analysis functions ready.")

Frame analysis functions ready.


---
##  Section 5 ; Language Model Summary Generation

CLIP gives us a list of labels per frame. The language model (FLAN-T5) stitches them into coherent, readable English sentences ; the way a human would describe what they saw.

In [18]:
def build_frame_context(frames_data: List[Dict], start_idx: int, end_idx: int) -> str:
 """
 Build a compact text description of a sequence of frames
 to feed into the language model.
 """
 lines = []
 for fd in frames_data[start_idx:end_idx+1]:
  top3 = ", ".join([lbl for lbl, _ in fd['labels'][:3]])
  lines.append(f"[{fd['timestamp_str']}] {top3}")
 return "\n".join(lines)


def generate_segment_description(
 context: str,
 start_ts: str,
 end_ts: str,
 max_new_tokens: int = 120
) -> str:
 """
 Ask FLAN-T5 to write a natural language description of a video segment
 given the CLIP observations as context.
 """
 prompt = f"""You are describing a video segment from {start_ts} to {end_ts}.
Based on these visual observations from the video frames:
{context}

Write 2-3 clear, specific sentences describing what is happening in this video segment.
Be concrete and descriptive. Focus on the actions, people, and setting."""

 inputs = tokenizer(
  prompt,
  return_tensors='pt',
  max_length=512,
  truncation=True).to(DEVICE)

 with torch.no_grad():
  outputs = lm_model.generate(
**inputs,
max_new_tokens=max_new_tokens,
num_beams=4,
temperature=0.7,
repetition_penalty=1.3,
early_stopping=True)

 return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def generate_overall_summary(all_descriptions: List[str], video_duration: float) -> str:
 """
 Generate a one-paragraph overall summary of the whole video.
 """
 combined = " ".join(all_descriptions[:8])  # use first 8 segment descriptions
 prompt= f"""This is a description of a {video_duration:.0f}-second video broken into segments:
{combined}

Write a single concise paragraph (3-4 sentences) summarising the entire video.
What is the video about? What is the main subject or storyline?"""

 inputs = tokenizer(
  prompt,
  return_tensors='pt',
  max_length=600,
  truncation=True).to(DEVICE)

 with torch.no_grad():
  outputs = lm_model.generate(
**inputs,
max_new_tokens=150,
num_beams=4,
temperature=0.7,
repetition_penalty=1.3,
early_stopping=True)
 return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(" Language model functions ready.")

 Language model functions ready.


---
## Section 6 ; Full Pipeline

This is the complete end-to-end function that takes a video path and returns a timestamped summary.

In [20]:
def analyse_video(
    video_path: str,
    frame_interval: float = 2.0,
    max_frames: int = 60,
    scene_threshold: float = 0.18,
    frames_per_segment: int = 5
) -> Dict:
    """
    Full pipeline of: video -> timestamped natural language summary.

    Steps:
    1. Extracts frames at regular intervals
    2. Runs CLIP on each frame -> label + confidence
    3. Detects scene changes via embedding similarity
    4. Groups frames into segments
    5. Generates a description per segment with FLAN-T5
    6. Generates one overall summary paragraph

    Returns a dict with all results for display + export.
    """
    total_start = time.time()

    print("=" * 60)
    print("VIDEO ANALYSIS PIPELINE")
    print("=" * 60)

    # ── Step 1: Extract frames ──────────────────────────────────────────────
    print("\n[1/5] Extracting frames...")

    frames_data = extract_frames(
        video_path,
        interval_seconds=frame_interval,
        max_frames=max_frames)

    if not frames_data:
        return {"error": "Could not extract frames from video."}

    video_duration = frames_data[-1]["timestamp"]

    # ── Step 2: CLIP analysis ───────────────────────────────────────────────
    print("\n[2/5] Running CLIP on each frame...")

    frames_data = analyse_frames(frames_data)

    # ── Step 3: Detect scene changes ────────────────────────────────────────
    print("\n[3/5] Detecting scene changes...")

    scene_cuts = detect_scene_changes(
        frames_data,
        threshold=scene_threshold)

    print(f"Detected {len(scene_cuts)} scene segments")

    # ── Step 4: Generate segment descriptions ───────────────────────────────
    print("\n[4/5] Generating natural language descriptions...")

    segments = []
    segment_boundaries = scene_cuts + [len(frames_data)]

    for seg_idx in range(len(scene_cuts)):

        start_i = segment_boundaries[seg_idx]
        end_i = min(
            segment_boundaries[seg_idx + 1] - 1,
            len(frames_data) - 1)

        start_ts = frames_data[start_i]["timestamp_str"]
        end_ts = frames_data[end_i]["timestamp_str"]

        context = build_frame_context(
            frames_data,
            start_i,
            end_i)

        description = generate_segment_description(
            context,
            start_ts,
            end_ts)

        # Collect dominant labels
        label_counter = {}

        for fd in frames_data[start_i:end_i + 1]:
            for lbl, score in fd["labels"][:3]:
                label_counter[lbl] = (
                    label_counter.get(lbl, 0) + score
                )

        top_labels = sorted(
            label_counter.items(),
            key=lambda x: -x[1]
        )[:4]

        avg_confidence = np.mean([
            fd["confidence"]
            for fd in frames_data[start_i:end_i + 1]
        ])

        segment = {
            "segment_id": seg_idx + 1,
            "start_ts": start_ts,
            "end_ts": end_ts,
            "start_frame_idx": start_i,
            "end_frame_idx": end_i,
            "num_frames": end_i - start_i + 1,
            "description": description,
            "top_labels": top_labels,
            "avg_confidence": avg_confidence,
        }

        segments.append(segment)

        print(
            f"Segment {seg_idx + 1:2d} "
            f"[{start_ts}–{end_ts}]: "
            f"{description[:70]}...")

    # ── Step 5: Overall summary ─────────────────────────────────────────────
    print("\n[5/5] Generating overall summary...")

    all_descs = [s["description"] for s in segments]

    overall_summary = generate_overall_summary(
        all_descs,
        video_duration)

    print(f"Summary: {overall_summary[:100]}...")

    total_time = time.time() - total_start

    realtime_factor = (
        video_duration / total_time
        if total_time > 0
        else 0)

    print("\n" + "=" * 60)
    print(f"ANALYSIS COMPLETE in {total_time:.1f}s")
    print(f"Video duration   : {video_duration:.1f}s")
    print(f"Processing time  : {total_time:.1f}s")
    print(
        f"Realtime factor  : "
        f"{realtime_factor:.1f}x "
        f"{'faster' if realtime_factor > 1 else '(slower than realtime)'}"
    )
    print(f"Frames processed : {len(frames_data)}")
    print(f"Scenes detected  : {len(segments)}")
    print("=" * 60)

    return {
        "frames_data": frames_data,
        "segments": segments,
        "overall_summary": overall_summary,
        "video_duration": video_duration,
        "total_time": total_time,
        "realtime_factor": realtime_factor,
        "num_frames": len(frames_data),
        "scene_cuts": scene_cuts,
    }
print("Full pipeline ready.")

Full pipeline ready.


---
## Section 7 ; Test with a Sample Video

We download a free, openly licensed video clip to test the pipeline end-to-end.

In [22]:
import urllib.request
import os

SAMPLE_VIDEO_URL = "https://samplelib.com/lib/preview/mp4/sample-30s.mp4"
SAMPLE_VIDEO_PATH = "/tmp/sample_video.mp4"

if not os.path.exists(SAMPLE_VIDEO_PATH):
    print("Downloading sample video...")

    urllib.request.urlretrieve(
        SAMPLE_VIDEO_URL,
        SAMPLE_VIDEO_PATH)

    print("Download complete.")

if os.path.exists(SAMPLE_VIDEO_PATH):
    size_mb = os.path.getsize(SAMPLE_VIDEO_PATH) / 1e6
    print(f"File size: {size_mb:.1f} MB")
else:
    print("Download failed.")

Download complete.
File size: 21.7 MB


In [23]:
import os

print(os.path.exists("/tmp/sample_video.mp4"))
print(os.path.getsize("/tmp/sample_video.mp4") / 1e6, "MB")

True
21.657943 MB


In [24]:
# ──  Run the full pipeline on the sample video ──────────────────────────────
results = analyse_video(
 video_path = SAMPLE_VIDEO_PATH,
 frame_interval= 2.5,# sample one frame every 2.5 seconds
 max_frames = 40, # cap at 40 frames for speed
 scene_threshold  = 0.18,  # sensitivity for scene change detection
)

VIDEO ANALYSIS PIPELINE

[1/5] Extracting frames...
Video info:
FPS          : 30.0
Total frames : 911
Duration     : 30.4s (0.5 min)
Sampling     : 1 frame every 2.5s
Frames extracted: 13

[2/5] Running CLIP on each frame...

Analysing 13 frames with CLIP...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [ 10/13] 00:22 ; outdoor scene                                 (30.0%) [13.3 frames/s]
  [ 13/13] 00:30 ; person walking                                (29.2%) [16.0 frames/s]

CLIP analysis done in 0.8s

[3/5] Detecting scene changes...
Detected 1 scene segments

[4/5] Generating natural language descriptions...
Segment  1 [00:00–00:30]: A person walks through a wooded area....

[5/5] Generating overall summary...
Summary: A person walks through a wooded area....

ANALYSIS COMPLETE in 20.0s
Video duration   : 30.0s
Processing time  : 20.0s
Realtime factor  : 1.5x faster
Frames processed : 13
Scenes detected  : 1


---
## Section 8 ; Visualisations

In [25]:
# ── Pretty-print the timestamped summary ─────────────────────────────────────
print("\n" + "═"*65)
print(" FULL VIDEO SUMMARY")
print("═"*65)
print(f"\n OVERALL:")
for line in textwrap.wrap(results['overall_summary'], width=62):
 print(f"  {line}")

print("\n TIMESTAMPED BREAKDOWN:")
print("  " + "-"*63)
for seg in results['segments']:
 print(f"\n  [{seg['start_ts']} -> {seg['end_ts']}]  Segment {seg['segment_id']}")
 for line in textwrap.wrap(seg['description'], width=60):
  print(f"  {line}")
 labels_str = " · ".join([f"{lbl[:25]}" for lbl, _ in seg['top_labels'][:3]])
 print(f" Tags: {labels_str}")
 print(f" Avg CLIP confidence: {seg['avg_confidence']*100:.1f}%")
print("\n " + "═"*63)


═════════════════════════════════════════════════════════════════
 FULL VIDEO SUMMARY
═════════════════════════════════════════════════════════════════

 OVERALL:
  A person walks through a wooded area.

 TIMESTAMPED BREAKDOWN:
  ---------------------------------------------------------------

  [00:00 -> 00:30]  Segment 1
  A person walks through a wooded area.
 Tags: outdoor scene · person walking · forest or nature
 Avg CLIP confidence: 29.9%

 ═══════════════════════════════════════════════════════════════


In [36]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Force notebook renderer
pio.renderers.default = "notebook_connected"

# Validate results
if "frames_data" not in results or len(results["frames_data"]) == 0:
    raise ValueError(
        "No frame data found. Run analyse_video() first.")

# Building dataframe
frames_df = pd.DataFrame([
    {
        "timestamp": fd["timestamp"],
        "timestamp_str": fd["timestamp_str"],
        "top_label": fd["top_label"],
        "confidence": fd["confidence"],
    }
    for fd in results["frames_data"]])

print("Frames dataframe shape:", frames_df.shape)
display(frames_df.head())

# Scene cut timestamps
scene_ts = [
    results["frames_data"][i]["timestamp"]
    for i in results["scene_cuts"]
]

# Create figure
fig1 = go.Figure()

fig1.add_trace(
    go.Scatter(
        x=frames_df["timestamp"],
        y=frames_df["confidence"],
        mode="lines+markers",
        line=dict(width=2),
        marker=dict(size=6),
        fill="tozeroy",
        name="CLIP Confidence",
        customdata=frames_df["top_label"],
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Confidence: %{y:.1%}"
            "<extra></extra>"
        ),
    )
)

# Scene cut markers
for ts in scene_ts[1:]:
    fig1.add_vline(
        x=ts,
        line_dash="dash",
        line_width=1.5,
        annotation_text="scene cut",
        annotation_position="top",
        annotation_font_size=9,)

max_conf = max(frames_df["confidence"])

fig1.update_layout(
    title="CLIP Confidence Timeline — Scene Changes Detected",
    xaxis_title="Timestamp (seconds)",
    yaxis_title="CLIP Confidence Score",
    yaxis=dict(
        tickformat=".0%",
        range=[0, max_conf * 1.3]
    ),
    template="plotly_dark",
    height=450,
    showlegend=True,)

# Displaying
fig1.show()

# Backup display method
fig1

Frames dataframe shape: (13, 4)


,timestamp,timestamp_str,top_label,confidence
0,0.0,00:00,forest or nature,0.313721
1,2.5,00:02,outdoor scene,0.231689
2,5.0,00:05,forest or nature,0.221558
3,7.5,00:07,outdoor scene,0.257080
4,10.0,00:10,outdoor scene,0.320312


In [27]:
# ── Plot: Scene segment Gantt chart ─────────────────────────────────────────
segs = results['segments']

# Parse timestamps to seconds for plotting
def ts_to_sec(ts: str) -> float:
 parts = ts.split(':')
 return int(parts[0]) * 60 + int(parts[1])

color_palette = px.colors.qualitative.Plotly

fig2 = go.Figure()
for i, seg in enumerate(segs):
 start_s = ts_to_sec(seg['start_ts'])
 end_s= ts_to_sec(seg['end_ts']) + 2  # +2 so tiny segments are visible
 color= color_palette[i % len(color_palette)]
 label_short = seg['top_labels'][0][0][:30] if seg['top_labels'] else 'unknown'

 fig2.add_trace(go.Bar(
  x=[end_s - start_s],
  y=[f"Seg {seg['segment_id']}"],
  base=start_s,
  orientation='h',
  marker_color=color,
  marker_opacity=0.85,
  name=f"Seg {seg['segment_id']}",
  hovertemplate=(
f"<b>Segment {seg['segment_id']}</b><br>"
f"{seg['start_ts']} -> {seg['end_ts']}<br>"
f"Main: {label_short}<br>"
f"Frames: {seg['num_frames']}<extra></extra>"
  ),
  showlegend=False
 ))

fig2.update_layout(
 title='Scene Segment Map ; Video Structure at a Glance',
 xaxis_title='Time (seconds)',
 barmode='overlay',
 template='plotly_dark', height=max(250, len(segs) * 35 + 100),
 paper_bgcolor='#0d0d1a', plot_bgcolor='#0d1117',
 font=dict(color='white', family='monospace'))
fig2.show()

In [28]:
# ── Plot 3: Top labels frequency across the whole video ───────────────────────
label_counts = {}
for fd in results['frames_data']:
 for lbl, score in fd['labels'][:3]:
  label_counts[lbl] = label_counts.get(lbl, 0) + 1

top_labels_overall = sorted(label_counts.items(), key=lambda x: -x[1])[:15]
lbl_names = [l[:40] for l, _ in top_labels_overall]
lbl_counts = [c for _, c in top_labels_overall]

fig3 = go.Figure(go.Bar(
 x=lbl_counts[::-1],
 y=lbl_names[::-1],
 orientation='h',
 marker=dict(
  color=lbl_counts[::-1],
  colorscale='Viridis',
  showscale=True),
 text=lbl_counts[::-1],
 textposition='outside'))
fig3.update_layout(
 title='Most Frequently Detected Labels Across All Frames',
 xaxis_title='Frame Count',
 template='plotly_dark', height=500,
 paper_bgcolor='#0d0d1a', plot_bgcolor='#0d1117',
 font=dict(color='white', family='monospace'),
 margin=dict(l=250))
fig3.show()

In [29]:
# ── Plot 4: PCA visualisation of frame embeddings (coloured by scene) ──────────
from sklearn.decomposition import PCA

embeddings = np.array([fd['embedding'] for fd in results['frames_data']])
pca  = PCA(n_components=2)
emb_2d  = pca.fit_transform(embeddings)

# Assign scene ID to each frame
scene_ids = np.zeros(len(results['frames_data']), dtype=int)
boundaries = results['scene_cuts'] + [len(results['frames_data'])]
for seg_idx in range(len(results['scene_cuts'])):
 start = boundaries[seg_idx]
 end= boundaries[seg_idx + 1]
 scene_ids[start:end] = seg_idx + 1

fig4 = go.Figure()
for scene_id in sorted(set(scene_ids)):
 mask = scene_ids == scene_id
 fig4.add_trace(go.Scatter(
  x=emb_2d[mask, 0],
  y=emb_2d[mask, 1],
  mode='markers+text',
  name=f'Scene {scene_id}',
  marker=dict(size=10, opacity=0.85),
  text=[results['frames_data'][i]['timestamp_str']
  for i in range(len(results['frames_data'])) if scene_ids[i] == scene_id],
  textposition='top center',
  textfont=dict(size=8)))

fig4.update_layout(
 title='PCA of CLIP Frame Embeddings ; Each Cluster = a Visual Scene',
 xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)',
 yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)',
 template='plotly_dark', height=500,
 paper_bgcolor='#0d0d1a', plot_bgcolor='#0d1117',
 font=dict(color='white', family='monospace'))
fig4.show()

---
## Section 9 ; Gradio Demo

**This is the interview demo.** Upload any video ; get a timestamped breakdown in your browser.

The demo runs entirely in Colab and serves a public URL via Gradio's tunnel.

In [45]:
# ── Dashboard Plot Functions ───────────────────────────────────────────────

def create_confidence_plot(results):

    import pandas as pd
    import plotly.graph_objects as go

    frames_df = pd.DataFrame([
        {
            "timestamp": fd["timestamp"],
            "confidence": fd["confidence"],
            "top_label": fd["top_label"]
        }
        for fd in results["frames_data"]
    ])

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=frames_df["timestamp"],
            y=frames_df["confidence"],
            mode="lines+markers",
            customdata=frames_df["top_label"],
            hovertemplate=(
                "<b>%{customdata}</b><br>"
                "Time: %{x:.1f}s<br>"
                "Confidence: %{y:.1%}"
                "<extra></extra>"),
            line=dict(width=3),
            marker=dict(size=8),
            fill="tozeroy"))

    for idx in results["scene_cuts"][1:]:

        ts = results["frames_data"][idx]["timestamp"]

        fig.add_vline(
            x=ts,
            line_dash="dash",
            annotation_text="Scene Cut")

    fig.update_layout(
        title="CLIP Confidence Timeline",
        xaxis_title="Time (seconds)",
        yaxis_title="Confidence",
        template="plotly_dark",
        height=450)

    return fig

def create_scene_timeline_plot(results):

    import plotly.graph_objects as go
    import plotly.express as px

    segs = results["segments"]

    def ts_to_sec(ts):
        mins, secs = ts.split(":")
        return int(mins) * 60 + int(secs)

    colors = px.colors.qualitative.Plotly

    fig = go.Figure()

    for i, seg in enumerate(segs):

        start_s = ts_to_sec(seg["start_ts"])
        end_s = ts_to_sec(seg["end_ts"]) + 2

        label_short = (
            seg["top_labels"][0][0][:30]
            if seg["top_labels"]
            else "unknown"
        )

        fig.add_trace(
            go.Bar(
                x=[end_s - start_s],
                y=[f"Scene {seg['segment_id']}"],
                base=start_s,
                orientation="h",
                marker_color=colors[i % len(colors)],
                showlegend=False,
                hovertemplate=(
                    f"<b>Scene {seg['segment_id']}</b><br>"
                    f"{seg['start_ts']} → {seg['end_ts']}<br>"
                    f"Main: {label_short}<br>"
                    f"Frames: {seg['num_frames']}"
                    "<extra></extra>"),)
        )

    fig.update_layout(
        title="Scene Timeline",
        template="plotly_dark",
        height=max(250, len(segs) * 35 + 100))

    return fig


def create_label_distribution_plot(results):

    import plotly.graph_objects as go

    label_counts = {}

    for fd in results["frames_data"]:

        for lbl, score in fd["labels"][:3]:

            # weighted by confidence
            label_counts[lbl] = (
                label_counts.get(lbl, 0)
                + score)

    top_labels = sorted(
        label_counts.items(),
        key=lambda x: -x[1])[:15]

    labels = [x[0][:40] for x in top_labels]
    scores = [x[1] for x in top_labels]

    fig = go.Figure(
        go.Bar(
            x=scores[::-1],
            y=labels[::-1],
            orientation="h",
            text=[
                f"{s:.2f}"
                for s in scores[::-1]
            ],
            textposition="outside"))

    fig.update_layout(
        title="Top Concepts Detected by CLIP",
        xaxis_title="Weighted Frequency",
        template="plotly_dark",
        height=500,
        margin=dict(l=250))

    return fig

def create_embedding_pca_plot(results):

    import numpy as np
    import plotly.graph_objects as go
    from sklearn.decomposition import PCA

    embeddings = np.array([
        fd["embedding"]
        for fd in results["frames_data"]])

    if len(embeddings) < 2:
        return go.Figure()

    pca = PCA(n_components=2)

    emb_2d = pca.fit_transform(
        embeddings)

    scene_ids = np.zeros(
        len(results["frames_data"]),
        dtype=int)

    boundaries = (
        results["scene_cuts"]
        + [len(results["frames_data"])])

    for seg_idx in range(
        len(results["scene_cuts"])
    ):

        start = boundaries[seg_idx]
        end = boundaries[seg_idx + 1]

        scene_ids[start:end] = seg_idx + 1

    fig = go.Figure()

    for scene_id in sorted(set(scene_ids)):

        mask = scene_ids == scene_id

        fig.add_trace(
            go.Scatter(
                x=emb_2d[mask, 0],
                y=emb_2d[mask, 1],
                mode="markers+text",
                name=f"Scene {scene_id}",
                text=[
                    results["frames_data"][i]["timestamp_str"]
                    for i in range(len(results["frames_data"]))
                    if scene_ids[i] == scene_id
                ],
                textposition="top center",
                marker=dict(size=10)))

    fig.update_layout(
        title="PCA of CLIP Embeddings",
        xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
        yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
        template="plotly_dark",
        height=500
    )

    return fig


print("Dashboard plots ready.")

Dashboard plots ready.


In [47]:
def format_results_for_gradio(results: Dict) -> Tuple[str, str]:
    """
    Format analysis results for Gradio display.
    """

    lines = []

    lines.append("OVERALL SUMMARY")
    lines.append("=" * 55)
    lines.append(results["overall_summary"])
    lines.append("")
    lines.append("TIMESTAMPED BREAKDOWN")
    lines.append("=" * 55)

    for seg in results["segments"]:

        lines.append(
            f"\n[{seg['start_ts']} -> {seg['end_ts']}]")

        lines.append(seg["description"])

        top_tags = " | ".join(
            [
                lbl[:25]
                for lbl, _ in seg["top_labels"][:3]
            ])

        lines.append(f"Tags: {top_tags}")

        lines.append(
            f"Confidence: {seg['avg_confidence'] * 100:.1f}%")

        lines.append("─" * 40)

    full_output = "\n".join(lines)

    stats_lines = [
        f"Video duration   : {results['video_duration']:.1f}s",
        f"Processing time  : {results['total_time']:.1f}s",
        f"Realtime factor  : {results['realtime_factor']:.1f}x",
        f"Frames processed : {results['num_frames']}",
        f"Scenes detected  : {len(results['segments'])}",
        f"Running on       : {DEVICE.upper()}",
    ]

    stats_output = "\n".join(stats_lines)

    return full_output, stats_output


def gradio_pipeline(
    video_file,
    frame_interval: float,
    max_frames: int,
    scene_sensitivity: float,
):
    """
    Gradio wrapper around the full analysis pipeline.
    """
    if video_file is None:

        return (
            "Please upload a video file.",
            "",
            None,
            None,
            None,
            None,)
    try:
        results = analyse_video(
            video_path=video_file,
            frame_interval=frame_interval,
            max_frames=int(max_frames),
            scene_threshold=scene_sensitivity,)

        if "error" in results:

            return (
                f"Error: {results['error']}",
                "",
                None,
                None,
                None,
                None,)

        summary_text, stats_text = (
            format_results_for_gradio(results))

        # ── Dashboard Charts ──────────────────────────

        confidence_fig = create_confidence_plot(
            results)

        scene_fig = create_scene_timeline_plot(
            results)

        labels_fig = create_label_distribution_plot(
            results)

        pca_fig = create_embedding_pca_plot(
            results)

        return (
            summary_text,
            stats_text,
            confidence_fig,
            scene_fig,
            labels_fig,
            pca_fig,)

    except Exception:

        import traceback

        return (
            f"Pipeline error:\n{traceback.format_exc()}",
            "",
            None,
            None,
            None,
            None,)

print("Gradio wrapper ready.")

Gradio wrapper ready.


In [48]:
# ── Launch the Gradio Dashboard ────────────────────────────────────────────
with gr.Blocks(
    title="Multimodal Video Understanding Dashboard",
    theme=gr.themes.Base(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate",
    ),
    css="""
    body { background: #0d0d1a; }

    .gradio-container {
        background: #0d0d1a;
        color: white;
        font-family: 'Courier New', monospace;
    }

    .gr-button-primary {
        background: #00D4FF !important;
        color: black !important;
        font-weight: bold;
    }

    h1, h2, h3 {
        color: #00D4FF !important;
    }

    .output-text {
        background: #0d1117 !important;
        border: 1px solid #00D4FF;
    }
    """
) as demo:

    # Header
    gr.Markdown("""
    # Multimodal Video Understanding Dashboard

    ### CLIP + FLAN-T5 + Semantic Scene Analysis

    Upload a video and receive:

    - Timestamped scene summaries
    - Scene segmentation
    - CLIP confidence analytics
    - Label distribution analytics
    - Embedding-space visualization
    - Overall video summary
    """)

    # Main Layout
    with gr.Row():

        # LEFT PANEL
        with gr.Column(scale=1):

            gr.Markdown("## Input")

            video_input = gr.Video(
                label="Upload Video",
                height=260)

            frame_interval_slider = gr.Slider(
                minimum=1.0,
                maximum=10.0,
                value=2.5,
                step=0.5,
                label="Frame Interval (seconds)")

            max_frames_slider = gr.Slider(
                minimum=10,
                maximum=80,
                value=40,
                step=5,
                label="Maximum Frames")

            scene_sensitivity_slider = gr.Slider(
                minimum=0.05,
                maximum=0.40,
                value=0.18,
                step=0.01,
                label="Scene Sensitivity")

            analyse_btn = gr.Button(
                "Analyse Video",
                variant="primary",
                size="lg")

            gr.Markdown("""
            ### Tips

            - Videos under 3 minutes work best
            - Lower frame intervals increase detail
            - GPU strongly recommended
            - Sports, tutorials, films and news work well
            """)

        # RIGHT PANEL
        with gr.Column(scale=2):

            summary_output = gr.Textbox(
                label="Timestamped Summary",
                lines=24,
                show_copy_button=True)

            stats_output = gr.Textbox(
                label="Performance Statistics",
                lines=8)

    # Dashboard Plots
    gr.Markdown("## Analytics Dashboard")

    with gr.Row():

        confidence_plot = gr.Plot(
            label="CLIP Confidence Timeline")

        scene_plot = gr.Plot(
            label="Scene Timeline")

    with gr.Row():

        labels_plot = gr.Plot(
            label="Label Distribution")

        pca_plot = gr.Plot(
            label="Embedding PCA")

    # Button Event
    analyse_btn.click(
        fn=gradio_pipeline,
        inputs=[
            video_input,
            frame_interval_slider,
            max_frames_slider,
            scene_sensitivity_slider,
        ],
        outputs=[
            summary_output,
            stats_output,
            confidence_plot,
            scene_plot,
            labels_plot,
            pca_plot,],)

    # Footer
    gr.Markdown("""
    ---

    ## Pipeline

    1. OpenCV extracts frames from the video
    2. CLIP generates semantic embeddings
    3. Zero-shot classification labels each frame
    4. Embedding similarity detects scene boundaries
    5. FLAN-T5 generates scene descriptions
    6. FLAN-T5 creates an overall summary
    7. Analytics dashboard visualizes model behavior

    ### Visualizations

    - Confidence Timeline
    - Scene Timeline
    - Label Distribution
    - Embedding PCA Projection
    """)

demo.launch(
    share=True,
    debug=False,
    quiet=True)

* Running on public URL: https://ffea68d7f9ff720ec7.gradio.live


---
## Section 10 ; Deep Dive: Score Custom Labels Against Any Frame

This section lets you run CLIP zero-shot classification with **your own custom labels** ; great for domain-specific tagging (e.g. sports events, medical footage, product demos).

In [35]:
def score_frame_against_custom_labels(
 frame_idx: int,
 custom_labels: List[str],
 frames_data: List[Dict]
):
 """
 Score a specific frame against any custom list of labels.
 Great for specialised use cases: medical, sports, retail, security, etc.
 """
 if frame_idx >= len(frames_data):
  print(f"Frame index {frame_idx} out of range (max: {len(frames_data)-1})")
  return

 fd  = frames_data[frame_idx]
 img = fd['frame']

 # Encode custom labels
 with torch.no_grad():
  tokens= clip.tokenize(custom_labels).to(DEVICE)
  txt_feat = clip_model.encode_text(tokens)
  txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

  img_tensor = clip_preprocess(img).unsqueeze(0).to(DEVICE)
  img_feat= clip_model.encode_image(img_tensor)
  img_feat= img_feat / img_feat.norm(dim=-1, keepdim=True)

 scores = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1)[0].cpu().numpy()

 print(f"\nFrame {frame_idx} at [{fd['timestamp_str']}]")
 print(f"Custom label scores:")
 for label, score in sorted(zip(custom_labels, scores), key=lambda x: -x[1]):
  bar = '█' * int(score * 400)
  print(f"  {label:<35} {score*100:5.1f}%  {bar}")

 # Visualise
 sorted_pairs = sorted(zip(custom_labels, scores), key=lambda x: x[1])
 fig = go.Figure(go.Bar(
  x=[s for _, s in sorted_pairs],
  y=[l for l, _ in sorted_pairs],
  orientation='h',
  marker=dict(
color=[s for _, s in sorted_pairs],
colorscale='Plasma',
showscale=True
  ),
  text=[f"{s*100:.1f}%" for _, s in sorted_pairs],
  textposition='outside'))
 fig.update_layout(
  title=f'Custom Label Scores ; Frame {frame_idx} [{fd["timestamp_str"]}]',
  xaxis_title='CLIP Confidence',
  template='plotly_dark', height=max(300, len(custom_labels)*35 + 100),
  paper_bgcolor='#0d0d1a', plot_bgcolor='#0d1117',
  font=dict(color='white'), margin=dict(l=200))
 fig.show()


# ── Example: sports-specific labels ─────────────────────────────────────────
SPORTS_LABELS = [
 "goal or score being made",
 "player running with ball",
 "crowd cheering in stadium",
 "referee making a call",
 "half-time break",
 "player injury or medical attention",
 "replay or slow motion",
 "trophy or celebration"]

# Score the 5th frame against sports labels (change frame_idx as needed)
score_frame_against_custom_labels(
 frame_idx = min(4, len(results['frames_data'])-1),
 custom_labels= SPORTS_LABELS,
 frames_data  = results['frames_data'])


Frame 4 at [00:10]
Custom label scores:
  half-time break                      27.1%  ████████████████████████████████████████████████████████████████████████████████████████████████████████████
  replay or slow motion                25.5%  █████████████████████████████████████████████████████████████████████████████████████████████████████
  player running with ball             19.5%  ██████████████████████████████████████████████████████████████████████████████
  goal or score being made             13.0%  ████████████████████████████████████████████████████
  referee making a call                 8.5%  ██████████████████████████████████
  trophy or celebration                 3.4%  █████████████
  player injury or medical attention    1.6%  ██████
  crowd cheering in stadium             1.3%  █████


---
## 🏁 Section 11 ; Interview Cheat Sheet

```
╔══════════════════════════════════════════════════════════════════╗
║ INTERVIEW CHEAT SHEET ; PROJECT 2 ║
╠══════════════════════════════════════════════════════════════════╣
║ ONE-LINER  ║
║I built an AI that watches any video clip and generates a  ║
║timestamped, plain-English summary of everything in it. ║
╠══════════════════════════════════════════════════════════════════╣
║ TECHNICAL ANSWER ║
║Multimodal pipeline: OpenCV extracts frames -> CLIP encodes║
║each frame into a 512-dim semantic vector -> zero-shot║
║classification over 80+ labels -> cosine distance detects  ║
║scene cuts -> FLAN-T5 writes natural language descriptions  ║
║per segment. Runs in under 60s on a free Colab T4 GPU.  ║
╠══════════════════════════════════════════════════════════════════╣
║ WHY CLIP║
║CLIP is trained on 400M image-text pairs. It understands║
║visual concepts in natural language ; no fine-tuning needed.  ║
║Zero-shot means it works on any video domain out of the box.  ║
╠══════════════════════════════════════════════════════════════════╣
║ BUSINESS STAKES  ║
║YouTube: 500 hours of video uploaded per minute║
║Netflix: 15,000+ titles need metadata tagging  ║
║Tesla: dashcams generate terabytes of footage daily  ║
║Apple: Photos.app needs to understand your home videos  ║
╠══════════════════════════════════════════════════════════════════╣
║ HONEST LIMITATION║
║FLAN-T5 occasionally hallucinates details not in the frames.  ║
║Next step: use a vision-language model (LLaVA or GPT-4V)  ║
║that takes the actual frame pixel data as input rather  ║
║than just CLIP's label predictions.║
╚══════════════════════════════════════════════════════════════════╝
```

---
*Built with: Python · OpenAI CLIP · Google FLAN-T5 · OpenCV · Gradio · Plotly · PyTorch*